In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.hist(df['Delivery_Time'])
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID", axis=1)
df

In [ ]:
# Task 2: Write your code here:
print(df.isnull().sum())

""" i want to do combination of fillna and dropna, but logically the most NaNs values
# -are in the target column "Delivery_Time" and it'll be a leakage if i fillna
# -using median/mean to fill values of the target since it's the most hurt and
# -has most missing values, so i decide to only drop rows of Delivery_Time, and fill the rest..
"""

# df['Weather'] = df['Weather'].fillna(df['Weather'].mean())                   # Cat. column
# df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mean()) # Cat. column
# df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mean())       # Cat. column
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

# Drop null values in the target "Delivery_Time"
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())

df.head()

# i got TypeError: can only concatenate str (not "int") to str, i realised i have to label encode cat columns before fillna.
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

print(df.head())
print(df.isnull().sum())

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
if duplicates > 0:
  print("Dropping Duplicates...")
  df.drop_duplicates(inplace=True)
  print("Duplicates Dropped.")
else:
  print("No Duplicate Samples Found.")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

categorical_cols = ['Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_cols = ["Distance_km", "Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type", "Preparation_Time_min", "Courier_Experience_yrs"]
df[features_cols] = scaler.fit_transform(df[features_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
df["Delivery_Time"].hist() # it's balanced data.

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]

print(X.shape, y.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold # is the correct one :)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
model = RandomForestRegressor()

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))

print(f"Avg MAE loss: {(sum(mae_scores) / 5)}") # i divide the total MAE losses on the number of Folds = 5

In [ ]:
# Task 1: Write your code here:
features_for_importance = [col for col in features_cols if col != 'is_legendary']
feature_importance = pd.DataFrame({
    'feature': features_for_importance,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
pd.DataFrame(y_pred).hist()

In [ ]:
# Task Bonus: Write your code here: